# STTran · 300 videos · Action Genome (zipped annotations + frames)

This notebook **downloads your repo on Colab** (GitHub zip first — no `git` username prompt), downloads **GloVe + Faster R-CNN + STTran weights**, merges **`annotations.zip`** and **`frames.zip`** into a valid AG tree, then runs **`run_first5_videos_all_frames.py`** with **`VIDEO_LIMIT=300`**.

**Outputs** (under `STTran/output/`):
- `logs/first5_videos/<VIDEO>.mp4.log` — full terminal-style log (parse for graphs / stats).
- `first5_videos/<VIDEO>.mp4/` — per-frame PNGs, GIF, legends.

After the run, **`export_graphs_json.py`** writes **`graphs.json`** in each video folder for programmatic graph extraction.

Push this repo (including `colab_minimal/`) to **GIT_REPO**, then edit the **config cell** paths (`GIT_REPO`, Drive zip paths). Use a **GPU** runtime before running.



In [ ]:
# --- EDIT THESE ---
GIT_REPO = "https://github.com/TommasoAiello08/STTran.git"
GIT_BRANCH = "main"  # change if your default branch differs

# Two zips on Google Drive (paths must exist after mounting Drive):
ANNOTATIONS_ZIP = "/content/drive/MyDrive/AG_zips/annotations.zip"
FRAMES_ZIP = "/content/drive/MyDrive/AG_zips/frames.zip"

# Where to build the Action Genome tree (ephemeral local disk on Colab):
AG_ROOT = "/content/ag_data"

# STTran code + outputs 
REPO_ROOT = "/content/STTran"

# How many videos to process (first N in frame_list order that appear in the chosen split)
VIDEO_LIMIT = 300
SPLIT = "test"   # "train" or "test"

# Optional: zip results back to Drive when done
OUTPUT_ZIP_ON_DRIVE = "/content/drive/MyDrive/sttran_outputs_300.zip"
# If your fork is private, set a classic PAT (repo scope) before the clone cell, e.g.:
#   import os; os.environ["GITHUB_TOKEN"] = "ghp_...."   # or use Colab secrets
# Public fork: leave unset.

# --- Optional: fast I/O (recommended for large frame sets) ---
# Colab's /content disk is much faster than reading from Drive. Options:
# (A) If you already have a full Action Genome tree on Drive, set DRIVE_AG_ROOT and
#     set USE_COPY_AG_FROM_DRIVE = True in the next cell to copy it to AG_ROOT.
# (B) If you only have zips on Drive, set COPY_ZIPS_TO_LOCAL = True in the next cell
#     to copy them to /content first, then unzip (faster + stable).
DRIVE_AG_ROOT = ""  # e.g. "/content/drive/MyDrive/challenge_4/ag_data" with annotations/ + frames/
COPY_ZIPS_TO_LOCAL = True   # set True (recommended) if zips are large — unzip from /content is faster
LOCAL_ZIP_DIR = "/content/ag_zips"  # temp dir for (B)
USE_COPY_AG_FROM_DRIVE = False  # (A) set True to copytree DRIVE_AG_ROOT -> AG_ROOT and skip zips




In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


In [ ]:
# Optional: copy Drive -> /content for faster I/O (same idea as flickr_ch4 example).
# See config: COPY_ZIPS_TO_LOCAL or USE_COPY_AG_FROM_DRIVE.
import os, shutil
from pathlib import Path

if USE_COPY_AG_FROM_DRIVE:
    assert DRIVE_AG_ROOT, "Set DRIVE_AG_ROOT to the Drive folder that contains annotations/ and frames/"
    src = Path(DRIVE_AG_ROOT)
    assert (src / "annotations").is_dir() and (src / "frames").is_dir(), src
    dst = Path(AG_ROOT)
    if dst.exists():
        shutil.rmtree(dst)
    print(f"Copying Action Genome tree (time depends on size)...\n  {src} -> {dst}")
    shutil.copytree(src, dst)
    print("Done. Next cell will skip zip merge.")
elif COPY_ZIPS_TO_LOCAL:
    Path(LOCAL_ZIP_DIR).mkdir(parents=True, exist_ok=True)
    la = Path(LOCAL_ZIP_DIR) / Path(ANNOTATIONS_ZIP).name
    lb = Path(LOCAL_ZIP_DIR) / Path(FRAMES_ZIP).name
    if not la.is_file() or la.stat().st_size != Path(ANNOTATIONS_ZIP).stat().st_size:
        print("Copying annotations zip to local SSD...")
        shutil.copy2(ANNOTATIONS_ZIP, la)
    else:
        print("Reuse local:", la)
    if not lb.is_file() or lb.stat().st_size != Path(FRAMES_ZIP).stat().st_size:
        print("Copying frames zip to local SSD (can take a while if huge)...")
        shutil.copy2(FRAMES_ZIP, lb)
    else:
        print("Reuse local:", lb)
    ANNOTATIONS_ZIP = str(la)
    FRAMES_ZIP = str(lb)
    print("Unzip cell will use:", ANNOTATIONS_ZIP, FRAMES_ZIP)
else:
    print("Using Drive paths directly (slower). Enable COPY_ZIPS_TO_LOCAL in config for faster unzip.")


In [ ]:
import json as _json
import os, re, shutil, subprocess, sys, tempfile, urllib.error, urllib.request, zipfile
from pathlib import Path


def run(cmd, cwd=None, env=None):
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=cwd, env=env)


def _rm_repo():
    p = Path(REPO_ROOT)
    if p.exists():
        shutil.rmtree(p, ignore_errors=True)
    subprocess.run(["rm", "-rf", REPO_ROOT], capture_output=True)


def _auth_headers():
    """Optional PAT for private repos (otherwise GitHub returns 404 on zip + API)."""
    h = {
        "User-Agent": "Mozilla/5.0 (compatible; Colab-STTran/1)",
        "Accept": "application/vnd.github+json",
    }
    tok = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    if tok:
        h["Authorization"] = f"token {tok}"
    return h


def _git_env():
    e = os.environ.copy()
    e["GIT_TERMINAL_PROMPT"] = "0"
    e.pop("GIT_ASKPASS", None)
    return e


def _github_slug(url):
    m = re.search(r"github\.com[:/]([^/]+)/([^/\s]+?)(?:\.git)?(?:/|\?|$)", url)
    if not m:
        return None
    return f"{m.group(1)}/{m.group(2)}"


def _http_get_bin(url: str) -> bytes:
    req = urllib.request.Request(url, headers=_auth_headers())
    with urllib.request.urlopen(req, timeout=120) as resp:
        return resp.read()


def _api_default_branch(slug: str):
    """Public: works without token. Private: set GITHUB_TOKEN or get 404."""
    api = f"https://api.github.com/repos/{slug}"
    req = urllib.request.Request(api, headers=_auth_headers())
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            data = _json.loads(r.read().decode())
        b = data.get("default_branch")
        if b:
            print("GitHub API default_branch:", b)
            return b
    except urllib.error.HTTPError as e:
        body = e.read().decode(errors="replace")[:500]
        print("GitHub API error:", e.code, body)
    except Exception as ex:
        print("GitHub API exception:", ex)
    return None


def _branches_to_try():
    seen = []
    for b in (
        GIT_BRANCH,
        _api_default_branch(_github_slug(GIT_REPO)) if _github_slug(GIT_REPO) else None,
        "main",
        "master",
    ):
        if b and b not in seen:
            seen.append(b)
    return seen


def _zip_urls_for(slug: str, branch: str):
    return [
        f"https://codeload.github.com/{slug}/zip/refs/heads/{branch}",
        f"https://github.com/{slug}/archive/refs/heads/{branch}.zip",
        f"https://github.com/{slug}/archive/{branch}.zip",
    ]


def _download_zipball() -> bool:
    slug = _github_slug(GIT_REPO)
    if not slug:
        print("Could not parse owner/repo from GIT_REPO:", GIT_REPO)
        return False
    _rm_repo()

    try_branches = _branches_to_try()
    print("Will try branches (in order):", try_branches)

    tmp = Path(tempfile.mkdtemp())
    zpath = tmp / "src.zip"
    last_err = None

    for branch in try_branches:
        for url in _zip_urls_for(slug, branch):
            try:
                print("Downloading:", url, flush=True)
                data = _http_get_bin(url)
                zpath.write_bytes(data)
                if zpath.stat().st_size < 2000:
                    print("  response too small; next URL")
                    continue
                break
            except Exception as ex:
                last_err = ex
                print("  failed:", ex)
        else:
            continue
        break
    else:
        shutil.rmtree(tmp, ignore_errors=True)
        print("All zip URLs failed. Last error:", last_err)
        print(
            "Hint: 404 often means wrong branch OR private repo without token.\n"
            "  • Set os.environ['GITHUB_TOKEN'] = '<PAT>' (classic PAT with repo scope).\n"
            "  • Or make the fork public / confirm GIT_REPO and GIT_BRANCH."
        )
        return False

    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(tmp)
    top = [p for p in tmp.iterdir() if p.is_dir() and p.name not in ("__MACOSX",)]
    if len(top) != 1:
        print("Unexpected archive layout:", [p.name for p in tmp.iterdir()])
        shutil.rmtree(tmp, ignore_errors=True)
        return False
    shutil.move(str(top[0]), REPO_ROOT)
    shutil.rmtree(tmp, ignore_errors=True)
    print("OK: GitHub zipball ->", REPO_ROOT)
    return True


def _authenticated_git_url():
    """Embed token in HTTPS URL so git never prompts (private repos)."""
    tok = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    if not tok:
        return GIT_REPO
    u = GIT_REPO.strip()
    if u.startswith("https://") and "@" not in u.split("//", 1)[1]:
        return u.replace("https://", f"https://x-access-token:{tok}@", 1)
    return u


def _git_clone_fallback():
    ge = _git_env()
    url = _authenticated_git_url()
    cmd_branch = [
        "git",
        "-c",
        "credential.helper=",
        "clone",
        "--depth",
        "1",
        "-b",
        GIT_BRANCH,
        url,
        REPO_ROOT,
    ]
    print("+ git clone -b", GIT_BRANCH, "->", REPO_ROOT)
    r = subprocess.run(cmd_branch, capture_output=True, text=True, env=ge)
    if r.returncode == 0:
        return True
    print("Git stderr:", r.stderr or r.stdout or "(empty)")
    _rm_repo()
    cmd_def = [
        "git",
        "-c",
        "credential.helper=",
        "clone",
        "--depth",
        "1",
        url,
        REPO_ROOT,
    ]
    print("+ git clone (default branch) ...")
    r2 = subprocess.run(cmd_def, capture_output=True, text=True, env=ge)
    if r2.returncode == 0:
        return True
    print("Git stderr:", r2.stderr or r2.stdout or "(empty)")
    return False


_rm_repo()

if not _download_zipball():
    print("Zipball failed — trying git with optional token in URL.")
    if not _git_clone_fallback():
        raise RuntimeError(
            "Could not fetch repo. For a private fork: before this cell run "
            "os.environ['GITHUB_TOKEN']='ghp_...' (classic PAT, repo scope). "
            "Check GIT_REPO and that the repo exists."
        )

if (Path(REPO_ROOT) / ".git").is_dir():
    print(subprocess.check_output(["git", "log", "-1", "--oneline"], cwd=REPO_ROOT).decode())
else:
    print("(fetched via zip — no .git; source tree is complete for running STTran)")

os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)




In [ ]:
# Merge annotations.zip + frames.zip -> AG_ROOT/{annotations,frames}
from pathlib import Path
import importlib.util

if USE_COPY_AG_FROM_DRIVE:
    assert (Path(AG_ROOT) / "annotations" / "frame_list.txt").is_file(), "Expected copied AG tree"
    print("Skipping zip merge (already copied from Drive). AG_ROOT:", AG_ROOT)
else:
    from pathlib import Path
    import importlib.util

    ann = Path(ANNOTATIONS_ZIP)
    frm = Path(FRAMES_ZIP)
    assert ann.is_file(), f"Missing {ann}"
    assert frm.is_file(), f"Missing {frm}"

    spec = importlib.util.spec_from_file_location(
        "prepare_ag_zips",
        Path(REPO_ROOT) / "colab_minimal" / "prepare_ag_zips.py",
    )
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    mod.prepare_action_genome(ann, frm, Path(AG_ROOT), clean=True)
    print("AG_ROOT ready:", AG_ROOT)


In [ ]:
# Install deps, compile extensions, GloVe, download official STTran + Faster R-CNN weights
run([sys.executable, "setup_colab.py"], cwd=REPO_ROOT)


In [ ]:
import os
os.environ["AG_DATA_PATH"] = AG_ROOT
os.environ["STTRAN_CKPT"] = str(Path(REPO_ROOT) / "ckpts" / "sttran_predcls.tar")
os.environ["SPLIT"] = SPLIT
os.environ["VIDEO_LIMIT"] = str(VIDEO_LIMIT)
os.environ["TOPK_PER_GROUP"] = "4"
os.environ["EDGE_THRESH"] = "0.0"
os.environ["VIZ_LAYOUT"] = "circular"
os.environ["VIZ_REUSE_LAYOUT"] = "1"
os.environ["MAX_RELS"] = "2000"

for k in ("AG_DATA_PATH", "VIDEO_LIMIT", "SPLIT"):
    print(k, os.environ.get(k))


In [ ]:
# Main run (~long): one folder per video under output/first5_videos/<VIDEO>.mp4/
run([sys.executable, "-u", "run_first5_videos_all_frames.py"], cwd=REPO_ROOT)


In [ ]:
# Parse logs -> graphs.json in each video folder (for downstream analysis)
import os
from pathlib import Path
os.environ.setdefault("TOPK_SPATIAL", "4")
os.environ.setdefault("TOPK_CONTACT", "4")
run([sys.executable, "export_graphs_json.py"], cwd=REPO_ROOT)


In [ ]:
# Optional: pack output/ into one zip on Drive (logs + viz + graphs.json)
from pathlib import Path
import shutil
out_dir = Path(REPO_ROOT) / "output"
if not out_dir.is_dir():
    print("No output directory — skipping zip.")
else:
    z = Path(OUTPUT_ZIP_ON_DRIVE)
    if z.suffix.lower() != ".zip":
        z = z.with_suffix(".zip")
    base = str(z.with_suffix(""))
    shutil.make_archive(base, "zip", root_dir=str(Path(REPO_ROOT)), base_dir="output")
    print("Wrote:", z)
